In [ ]:
import pandas as pd

ident = pd.read_csv("../data/raw/GSE133344_filtered_cell_identities.csv.gz")
print(ident.shape)
print(ident.head())
print(ident.columns.tolist())

# col = cell_barcode is the key for joining with adata.obs (metadata) later


In [ ]:
import scanpy as sc

adata = sc.read_mtx("../data/raw/GSE133344_filtered_matrix.mtx.gz").T
# the barcodes file was probably made by cellranger aggr, after indiv cellranger runs per lane.  barcodes already have laneID appended
barcodes = pd.read_csv("../data/raw/GSE133344_filtered_barcodes.tsv.gz", header=None)[0]
genes = pd.read_csv("../data/raw/GSE133344_filtered_genes.tsv.gz", header=None, sep="\t")

# adata.obs is metadata, indexed by barcode-laneID.
# adata.var is metadata for genes.  when we load genes[1].values into indices for var, it doesn't yet have any columns

adata.obs_names = barcodes.values
adata.var_names = genes[1].values  # gene symbol column; check genes.head() if this looks wrong
adata.var_names_make_unique()

print(adata)
print(adata.obs_names[:5].tolist())

In [ ]:
# ident was loaded in in the first cell
# ident = pd.read_csv("../data/raw/GSE133344_filtered_cell_identities.csv.gz")

ident_idx = ident.set_index("cell_barcode")
adata.obs = adata.obs.join(ident_idx, how="left")
# join in py is done by index
# adata.obs is already indexed by barcode (which is already appended with lane ID since barcodes were repeated across lanes)

print(adata.obs["guide_identity"].isna().sum(), "cells with no guide call")
print(adata.obs.shape)
adata.obs.head()

In [ ]:
# only 223 cells have no guide call; there are 111k rows (cells) - this is 0.2% - safe to just drop

adata = adata[adata.obs["guide_identity"].notna()].copy()
print(adata.shape)

In [ ]:
# look at good_coverage (metric decided by authors) and number_of_cells for more QC

print(adata.obs["good_coverage"].value_counts())
print(adata.obs["number_of_cells"].value_counts())

In [ ]:
# there are 6.9k cells with bad coverage
# the number_of_cells is correlated with sets that got the same set of CRISPR guides delivered (effectively, this is probably "cell replicates")
# so, likely the bad coverage cells are all in unique "categories" with 0 cells assigned to them
# drop the cells with bad coverage.
# adata.obs good coverage is actually an object, not true Bool values. convert first.

mask = adata.obs["good_coverage"].astype(bool).values
adata = adata[mask].copy()
print(adata.shape)

In [ ]:

# add a col to adata.var (which currently has none) for bool for "MT-*"
adata.var["mt"] = adata.var_names.str.startswith("MT-")
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], inplace=True, percent_top=None)

adata.obs[["n_genes_by_counts", "total_counts", "pct_counts_mt"]].describe()

In [ ]:
# make some figs for QC to save

import matplotlib.pyplot as plt

# plt.subplots returns (1) overall figure area (2) one or more axes objects - autocreates axes with these params inside of fig
fig, axes = plt.subplots(1, 3, figsize=(7, 4))
metrics = ["n_genes_by_counts", "total_counts", "pct_counts_mt"]
titles = ["Genes per cell", "UMIs per cell", "% mitochondrial counts"]
colors = ["#4C72B0", "#55A868", "#C44E52"]

for ax, m, t, c in zip(axes, metrics, titles, colors):
    ax.violinplot(adata.obs[m], showmedians=True)
    ax.set_title(t)
    ax.set_ylabel(m)
    ax.collections[0].set_facecolor(c)
    ax.collections[0].set_alpha(0.7)

fig.suptitle("Per-cell QC metrics (pre-filtering)")
fig.tight_layout()
fig.savefig("../results/figures/qc_violins.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# second plot: 
fig, ax = plt.subplots(figsize=(6, 5))
sc_plot = ax.scatter(
    adata.obs["total_counts"], adata.obs["pct_counts_mt"],
    s=4, alpha=0.3, c=adata.obs["n_genes_by_counts"], cmap="viridis"
)
ax.set_xlabel("Total UMI counts")
ax.set_ylabel("% mitochondrial counts")
ax.set_title("Mito fraction vs. library size")
cbar = fig.colorbar(sc_plot, ax=ax)
cbar.set_label("Genes per cell")
fig.tight_layout()
fig.savefig("../results/figures/qc_scatter_mito_vs_counts.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# these two graphs show that most cells are <10% mt, but there are some with very high (40%)
# and that these high mt% cells are also those with low total counts.
# these should be filtered. a common cutoff is 10-20% mt, but lets look at the data directly for where that tail appears
print(adata.obs["pct_counts_mt"].quantile([0.90, 0.95, 0.97, 0.98, 0.99]))
print(adata.obs["total_counts"].quantile([0.99, 0.995, 0.999]))
print(adata.obs["n_genes_by_counts"].quantile([0.99, 0.995, 0.999]))

In [ ]:
# Perturbation-agnostic single-cell QC filtering.
# Thresholds chosen from the actual distribution (.describe() + 
# .quantile() checks above), not generic defaults, since this matrix is
# already Cell Ranger cell-called (no empty droplets to worry about).

n_before = adata.n_obs

mask = (
    # High mito % = stressed/dying cells; tail departs sharply above ~10% (99th pctile), and these cells were concentrated at low total_counts in the scatter plot
    (adata.obs["pct_counts_mt"] < 10) &

    # Thin tail of likely doublets above ~35.8k (99.5th pctile); bulk of cells sit well below this
    (adata.obs["total_counts"] < 35000) &

    # Same doublet logic as counts; thin tail above ~5.3k (99.5th pctile)
    (adata.obs["n_genes_by_counts"] < 5500)
)

adata = adata[mask].copy()
print(f"{n_before} -> {adata.n_obs} cells ({n_before - adata.n_obs} removed)")

In [ ]:
# save the raw counts layer before modifying in place with normalization
adata.layers["counts"] = adata.X.copy()

# Normalize for per-cell sequencing depth (not gene length -- UMI counts
# don't need length correction the way fragment-based bulk RNA-seq does).
# target_sum=1e4 matches Seurat's CP10K convention.
sc.pp.normalize_total(adata, target_sum=1e4)

# Log-transform to stabilize variance / compress the heavy right skew
# typical of count data, standard pre-step for downstream analysis (DE, PCA).
sc.pp.log1p(adata)

# store the lognorm layer in adata
adata.layers["lognorm"] = adata.X.copy()